# Diagnóstico probabilístico del NPP-LSTM

## Objetivo

Analizar las diferencias entre las probabilidades estimadas
por NPP Softplus W20 y Poisson.

Se estudiarán:
- Sesgo de probabilidad media.
- Brier Skill Score respecto a Poisson.
- Casos donde NPP asigna probabilidades altas sin
  observarse el evento objetivo.
- Casos donde ocurre un evento objetivo pese a una
  probabilidad relativamente baja.

Evaluación retrospectiva, no operativa.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

raiz = Path.cwd()
if not (raiz / "results").exists():
    raiz = raiz.parent

pred = pd.read_csv(
    raiz / "results/predicciones_NPP_Poisson.csv"
)

filas = []

for dias in [1, 7, 30]:
    npp = pred[
        (pred["Horizonte_dias"] == dias) &
        (pred["Modelo"] == "NPP Softplus W20")
    ].sort_values("Referencia_horas")

    poisson = pred[
        (pred["Horizonte_dias"] == dias) &
        (pred["Modelo"] == "Poisson")
    ].sort_values("Referencia_horas")

    assert np.array_equal(
        npp["Referencia_horas"].to_numpy(),
        poisson["Referencia_horas"].to_numpy()
    )

    y = npp["Observado"].to_numpy()
    p_npp = npp["Probabilidad"].to_numpy()
    p_poisson = poisson["Probabilidad"].to_numpy()

    assert np.array_equal(
        y, poisson["Observado"].to_numpy()
    )

    brier_npp = np.mean((p_npp - y) ** 2)
    brier_poisson = np.mean((p_poisson - y) ** 2)

    filas.append({
        "Dias": dias,
        "Observado_pct": 100 * y.mean(),
        "NPP_pct": 100 * p_npp.mean(),
        "Poisson_pct": 100 * p_poisson.mean(),
        "Sesgo_NPP_pp": 100 * (p_npp.mean() - y.mean()),
        "Brier_Skill_pct": 100 * (
            1 - brier_npp / brier_poisson
        )
    })

diagnostico = pd.DataFrame(filas)

display(diagnostico.round(3))

diagnostico.to_csv(
    raiz / "results/diagnostico_probabilidades.csv",
    index=False
)

,Dias,Observado_pct,NPP_pct,Poisson_pct,Sesgo_NPP_pp,Brier_Skill_pct
0,1,8.557,15.957,8.236,7.400,-2.868
1,7,37.775,63.317,45.209,25.542,-19.346
2,30,75.646,96.838,92.411,21.192,-7.806


In [2]:
npp = pred[
    pred["Modelo"] == "NPP Softplus W20"
].copy()

# Probabilidad alta, pero no ocurrió el evento
falsas_expectativas = (
    npp[npp["Observado"] == 0]
    .sort_values("Probabilidad", ascending=False)
    .head(10)
)

# Ocurrió el evento, pero recibió baja probabilidad
eventos_subestimados = (
    npp[npp["Observado"] == 1]
    .sort_values("Probabilidad", ascending=True)
    .head(10)
)

print("Probabilidades altas sin evento observado:")
display(falsas_expectativas)

print("Eventos observados con probabilidad baja:")
display(eventos_subestimados)

Probabilidades altas sin evento observado:


,Horizonte_dias,Modelo,Referencia_horas,Observado,Probabilidad
3789,30,NPP Softplus W20,529593.275833,0,0.993203
3788,30,NPP Softplus W20,529588.600000,0,0.992681
3790,30,NPP Softplus W20,529697.533611,0,0.992644
3845,30,NPP Softplus W20,534244.495833,0,0.991937
3787,30,NPP Softplus W20,529583.373056,0,0.991734
3791,30,NPP Softplus W20,529763.271389,0,0.991323
3846,30,NPP Softplus W20,534253.373333,0,0.991159
3836,30,NPP Softplus W20,533418.542778,0,0.990096
3792,30,NPP Softplus W20,529803.234167,0,0.989858
3987,30,NPP Softplus W20,565136.819167,0,0.989567


Eventos observados con probabilidad baja:


,Horizonte_dias,Modelo,Referencia_horas,Observado,Probabilidad
706,1,NPP Softplus W20,564338.457222,1,0.095745
741,1,NPP Softplus W20,568429.281944,1,0.100206
599,1,NPP Softplus W20,542430.108333,1,0.100888
537,1,NPP Softplus W20,530978.617778,1,0.119120
559,1,NPP Softplus W20,533253.062222,1,0.119452
536,1,NPP Softplus W20,530968.247222,1,0.120572
571,1,NPP Softplus W20,534209.989444,1,0.122326
184,1,NPP Softplus W20,479909.632778,1,0.129166
196,1,NPP Softplus W20,482810.441389,1,0.129905
546,1,NPP Softplus W20,532226.889167,1,0.130709


In [3]:
filas = []

for (dias, nombre), grupo in pred.groupby(
    ["Horizonte_dias", "Modelo"]
):
    y = grupo["Observado"].to_numpy()
    p = grupo["Probabilidad"].to_numpy()

    # Regla binaria: probabilidad >= 50 %
    prediccion = (p >= 0.5).astype(int)

    tp = np.sum((prediccion == 1) & (y == 1))
    tn = np.sum((prediccion == 0) & (y == 0))
    fp = np.sum((prediccion == 1) & (y == 0))
    fn = np.sum((prediccion == 0) & (y == 1))

    accuracy = (tp + tn) / len(y)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    filas.append({
        "Dias": dias,
        "Modelo": nombre,
        "Accuracy_pct": 100 * accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    })

clasificacion = pd.DataFrame(filas)

display(clasificacion.round(4))

clasificacion.to_csv(
    raiz / "results/metricas_clasificacion.csv",
    index=False
)

,Dias,Modelo,Accuracy_pct,Precision,Recall,F1,TP,TN,FP,FN
0,1,NPP Softplus W20,91.4425,0.0000,0.0000,0.0000,0,748,0,70
1,1,Poisson,91.4425,0.0000,0.0000,0.0000,0,748,0,70
2,7,NPP Softplus W20,39.8533,0.3828,0.9676,0.5486,299,27,482,10
3,7,Poisson,62.2249,0.0000,0.0000,0.0000,0,509,0,309
4,30,NPP Softplus W20,75.6458,0.7565,1.0000,0.8613,615,0,198,0
5,30,Poisson,75.6458,0.7565,1.0000,0.8613,615,0,198,0
